# 04. Mean Squared Displacement and Non-Linear Chaos

Statistical analysis of active microswimmer dispersion and quantification of chaos via Finite-Time Lyapunov Exponents.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
from active_matter import simulate_ensemble, compute_msd, fit_diffusion_exponent, compute_finite_time_lyapunov, set_visual_style


### 1. Mean Squared Displacement Across Propulsion Velocities


In [ ]:
np.random.seed(42)
N = 500
X0 = np.random.uniform(0.0, 2*np.pi, size=(N, 3))
velocities = [0.0, 0.1, 0.3]

fig, ax = plt.subplots(figsize=(7, 4.5))
for vp in velocities:
    traj, times = simulate_ensemble(X0, dt=0.01, nt=4000, nsave=40, vp=vp, alpha=0.95, wrap_pbc=False)
    msd = compute_msd(traj)
    fit = fit_diffusion_exponent(times, msd, t_min=0.5, t_max=5.0)
    ax.loglog(times, msd, label=f" = {vp:.1f}$ (beta = {fit['beta']:.2f})")

ax.set_xlabel("Time (t)"); ax.set_ylabel("MSD")
ax.set_title("Ensemble MSD on Log-Log Scale")
ax.legend(); ax.grid(True, linestyle=":"); plt.show()


### 2. Trajectory Divergence and Lyapunov Exponent


In [ ]:
X_base = np.array([[1.0, 1.0, np.pi / 2.0]])
pert = 1e-5
X_pair = np.vstack((X_base, X_base + pert))

traj, times = simulate_ensemble(X_pair, dt=0.005, nt=1500, nsave=1, vp=1.0, alpha=0.0, wrap_pbc=False)
res = compute_finite_time_lyapunov(traj[:, 0, :], traj[:, 1, :], dt=0.005, t_max=4.0)

print(f"Finite-Time Lyapunov Exponent: {res['lyapunov_exponent']:.4f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(res['times'], np.log(res['separation'] / pert), color="#e63946", linewidth=2)
ax.set_xlabel("Time"); ax.set_ylabel(r"$\ln(|\delta(t)| / |\delta(0)|)$")
ax.set_title("Exponential Divergence in Phase Space")
ax.grid(True, linestyle=":"); plt.show()
